# From next-token model to chat assistant

A language-exchange app swapped in a base model by mistake, and its "Ask the tutor" box answered a learner's
question by writing more questions. In this notebook you replay the lesson's recorded replies from three
checkpoints of SmolLM2-135M (base, instruction-tuned, instruction- and preference-tuned), then build the chat
template yourself with `apply_chat_template` on the real tokenizer, and find out why the right model called the
wrong way still fails.

About 5 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no account and no
key. Only the tokenizer files are downloaded (a few MB). Colab may print a note that no `HF_TOKEN` is set; you
can ignore it.

## 1. Load the tokenizer of the tuned model

The lesson's model is `HuggingFaceTB/SmolLM2-135M-Instruct`. Its tokenizer holds the **chat template**. You need
no model weights to build the string the model reads: the tokenizer files are enough. Colab already has
transformers, so there is nothing to install.

**Look for:** the transformers version (the lesson ran 5.22.2; another recent version gives the same strings) and
the tokenizer's end-of-turn token, `<|im_end|>`.

In [1]:
import os, warnings
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
import transformers
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M-Instruct")
print("transformers:", transformers.__version__)
print("end-of-turn token:", tokenizer.eos_token)

transformers: 5.22.2
end-of-turn token: <|im_end|>


## 2. The recorded replies

The lesson's replies were made with Ollama (`/api/generate`, `raw: true`, temperature 0, seed 0, at most 120 new
tokens, stop on `<|im_end|>`). You have no Ollama server here, so `RECORDED` holds the real responses, copied
exactly. `generate(model, prompt)` has the same shape as the lesson's code: it finds the recorded response for
this exact model and this exact prompt string. Nothing is random: the same request always gives the same reply.

`done_reason` is `"stop"` when the model wrote the end-of-turn marker itself, and `"length"` when the 120-token
limit cut it off. `eval_count` is the number of tokens it wrote.

**Look for:** 9 recorded runs from 3 models.

In [2]:
MODELS = {
    "base": "hf.co/QuantFactory/SmolLM2-135M-GGUF:Q4_0",              # pretraining only
    "sft": "hf.co/mradermacher/smollm2-135M-SFT-Only-GGUF:IQ4_XS",    # + instruction tuning
    "instruct": "hf.co/bartowski/SmolLM2-135M-Instruct-GGUF:Q4_0",    # + instruction tuning + DPO
}

RECORDED = [
    {
        "run": "ser-base-plain",
        "model": "hf.co/QuantFactory/SmolLM2-135M-GGUF:Q4_0",
        "prompt": "When do I use ser and when estar?",
        "response": "\n1. What is the difference between a function and an expression?\n2. How can you tell if a function is defined or not?\n3. What are some examples of functions that have been used in the real world?\n4. What does it mean to be \"in the real world\"?\n5. What do we call the set of all functions?\n6. What is an example of a function that has no inverse?\n7. How can you tell if a function is defined or not?\n8. What are some examples of functions that have been used in the",
        "done_reason": "length", "eval_count": 120,
    },
    {
        "run": "ser-instruct-plain",
        "model": "hf.co/bartowski/SmolLM2-135M-Instruct-GGUF:Q4_0",
        "prompt": "When do I use ser and when estar?",
        "response": "",
        "done_reason": "stop", "eval_count": 1,
    },
    {
        "run": "gracias-base-plain",
        "model": "hf.co/QuantFactory/SmolLM2-135M-GGUF:Q4_0",
        "prompt": "What does gracias mean?",
        "response": "\n\nThe word “gracias” is a Latin phrase that means “kindness.” It’s the Spanish equivalent of the English words “merci,” which means thankfulness. The word also has other meanings, such as “gratitude or thanksgiving,” and it can be used to describe both kindnesses and gifts.\n\nThe word gracias is a very common expression in Spanish culture. It’s one of the most commonly used words in the world of business and industry. It’s also an important part of the culture, as well as its social and cultural",
        "done_reason": "length", "eval_count": 120,
    },
    {
        "run": "gracias-base-templated",
        "model": "hf.co/QuantFactory/SmolLM2-135M-GGUF:Q4_0",
        "prompt": "<|im_start|>system\nYou are a patient language tutor. Answer in one or two sentences.<|im_end|>\n<|im_start|>user\nWhat does gracias mean?<|im_end|>\n<|im_start|>assistant\n",
        "response": "Assistants and assistants, what is it called? Assisting\nAssists to the point of being an assistant\nActor/soparrader (or actor)what is it called? Assisted\nAssistance or assistance. What is it called? Assisting\nAssistant(soparrader), what does he mean by that? Assisting\nAssistant, what do you call him? Assisting\nActor/soparrader (or actor)what does he mean by that? Assisted\nAssistance or assistance. What is it called",
        "done_reason": "length", "eval_count": 120,
    },
    {
        "run": "gracias-sft-templated",
        "model": "hf.co/mradermacher/smollm2-135M-SFT-Only-GGUF:IQ4_XS",
        "prompt": "<|im_start|>system\nYou are a patient language tutor. Answer in one or two sentences.<|im_end|>\n<|im_start|>user\nWhat does gracias mean?<|im_end|>\n<|im_start|>assistant\n",
        "response": "\"Gratis\" is a Latin phrase that means \"thank you.\" It's often used to express gratitude for something, such as a gift, a visit, or a recommendation from the speaker. It's not just about showing appreciation; it's also about being kind and understanding in your response.",
        "done_reason": "stop", "eval_count": 61,
    },
    {
        "run": "gracias-instruct-templated",
        "model": "hf.co/bartowski/SmolLM2-135M-Instruct-GGUF:Q4_0",
        "prompt": "<|im_start|>system\nYou are a patient language tutor. Answer in one or two sentences.<|im_end|>\n<|im_start|>user\nWhat does gracias mean?<|im_end|>\n<|im_start|>assistant\n",
        "response": "Gracias means \"thank you\" in Spanish, and it's used to express gratitude towards someone for their kindness, support, or assistance. It can also be translated as \"good luck,\" \"fortunate circumstance,\" or simply \"thankful.\"",
        "done_reason": "stop", "eval_count": 51,
    },
    {
        "run": "gracias-instruct-plain",
        "model": "hf.co/bartowski/SmolLM2-135M-Instruct-GGUF:Q4_0",
        "prompt": "What does gracias mean?",
        "response": "",
        "done_reason": "stop", "eval_count": 1,
    },
    {
        "run": "yo-instruct-plain",
        "model": "hf.co/bartowski/SmolLM2-135M-Instruct-GGUF:Q4_0",
        "prompt": "Is this sentence correct: Yo soy cansado?",
        "response": "\nOptions are:\n1. No, it's not correct.\n2. Yes, it is correct.",
        "done_reason": "stop", "eval_count": 24,
    },
    {
        "run": "yo-instruct-templated",
        "model": "hf.co/bartowski/SmolLM2-135M-Instruct-GGUF:Q4_0",
        "prompt": "<|im_start|>system\nYou are a patient language tutor. Answer in one or two sentences.<|im_end|>\n<|im_start|>user\nIs this sentence correct: Yo soy cansado?<|im_end|>\n<|im_start|>assistant\n",
        "response": "Yes, that's the correct answer. The phrase \"yoy soy\" is an idiomatic expression used to express the idea of something being good or pleasant. It means \"I have a good taste in mind.\"",
        "done_reason": "stop", "eval_count": 44,
    },
]

def generate(model, prompt):
    """Replay: the recorded Ollama response for this exact model and prompt."""
    for r in RECORDED:
        if r["model"] == model and r["prompt"] == prompt:
            return r
    raise KeyError("No recorded reply for this model and this exact prompt string.")

def show(label, res):
    print(f"--- {label}")
    print(res["response"].strip("\n") or "(empty reply)")
    print(f"[done_reason: {res['done_reason']}, tokens: {res['eval_count']}]\n")

print("recorded runs:", len(RECORDED))
print("models:", len({r["model"] for r in RECORDED}))

recorded runs: 9
models: 3


## 3. The tutor that asked back

This is the request the app sent after the swap: the base model, and the learner's question as plain text.

**Look for:** a numbered list of new questions about functions, not one word about ser and estar. It ends
mid-sentence with `done_reason: length` after 120 tokens: the model never decided to stop.

In [3]:
question = "When do I use ser and when estar?"
show("base model, plain question", generate(MODELS["base"], question))

--- base model, plain question
1. What is the difference between a function and an expression?
2. How can you tell if a function is defined or not?
3. What are some examples of functions that have been used in the real world?
4. What does it mean to be "in the real world"?
5. What do we call the set of all functions?
6. What is an example of a function that has no inverse?
7. How can you tell if a function is defined or not?
8. What are some examples of functions that have been used in the
[done_reason: length, tokens: 120]



## 4. Four replies to one question

The team sent "What does gracias mean?" four ways. `G_T` is the full chat-format string, with a system line, the
user's turn, and an open assistant turn. You will build this same string with the library in step 9.

**Look for:** the two base-model replies run to `length` (120 tokens): one reads like an article, the other is
word salad about "assistant". The two tuned models reply and stop on their own (`stop`, 61 and 51 tokens). The SFT
model's answer is wrong ("Gratis"), but it is shaped as a reply.

In [4]:
G_Q = "What does gracias mean?"
G_T = ("<|im_start|>system\nYou are a patient language tutor. Answer in one or two sentences.<|im_end|>\n"
       "<|im_start|>user\nWhat does gracias mean?<|im_end|>\n<|im_start|>assistant\n")

show("B: base, plain question", generate(MODELS["base"], G_Q))
show("D: base, chat format", generate(MODELS["base"], G_T))
show("C: SFT, chat format", generate(MODELS["sft"], G_T))
show("A: instruct, chat format", generate(MODELS["instruct"], G_T))

--- B: base, plain question
The word “gracias” is a Latin phrase that means “kindness.” It’s the Spanish equivalent of the English words “merci,” which means thankfulness. The word also has other meanings, such as “gratitude or thanksgiving,” and it can be used to describe both kindnesses and gifts.

The word gracias is a very common expression in Spanish culture. It’s one of the most commonly used words in the world of business and industry. It’s also an important part of the culture, as well as its social and cultural
[done_reason: length, tokens: 120]

--- D: base, chat format
Assistants and assistants, what is it called? Assisting
Assists to the point of being an assistant
Actor/soparrader (or actor)what is it called? Assisted
Assistance or assistance. What is it called? Assisting
Assistant(soparrader), what does he mean by that? Assisting
Assistant, what do you call him? Assisting
Actor/soparrader (or actor)what does he mean by that? Assisted
Assistance or assistance. What is it c

## 5. Let the library build the string

This is the lesson's `template.py`, first call, line for line. You pass a `messages` list (the same shape a hosted
chat API takes) and the tokenizer turns it into its own template. `tokenize=False` returns the string so you can
read it. `repr` shows the newlines as `\n`.

**Look for:** the string starts with `<|im_start|>system` and ends with an open `<|im_start|>assistant` line, with
nothing after it. That open turn is where the model starts writing.

In [5]:
tutor = ("You are a patient Spanish tutor. "
         "Answer in two sentences.")
ask = "When do I use ser and when estar?"
messages = [
    {"role": "system", "content": tutor},
    {"role": "user", "content": ask},
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False,
    add_generation_prompt=True)
print(repr(prompt))
print()
print(prompt)

'<|im_start|>system\nYou are a patient Spanish tutor. Answer in two sentences.<|im_end|>\n<|im_start|>user\nWhen do I use ser and when estar?<|im_end|>\n<|im_start|>assistant\n'

<|im_start|>system
You are a patient Spanish tutor. Answer in two sentences.<|im_end|>
<|im_start|>user
When do I use ser and when estar?<|im_end|>
<|im_start|>assistant



## 6. Without the generation prompt

The same call with `add_generation_prompt=False`. You use this when you format finished conversations as training
data, not when you ask the model for a reply.

**Look for:** the last line is now the user's turn, `When do I use ser and when estar?<|im_end|>`. The open
assistant line is gone, so the model gets no cue that it is its turn.

In [6]:
no_gen = tokenizer.apply_chat_template(
    messages, tokenize=False,
    add_generation_prompt=False)
print(repr(no_gen))
print("ends with an open assistant turn:", no_gen.endswith("<|im_start|>assistant\n"))

'<|im_start|>system\nYou are a patient Spanish tutor. Answer in two sentences.<|im_end|>\n<|im_start|>user\nWhen do I use ser and when estar?<|im_end|>\n'
ends with an open assistant turn: False


## 7. When the system message is missing

`messages[1:]` is the list without its first item (like `messages.slice(1)` in JavaScript): only the learner's
message.

**Look for:** a system line you did not write: "You are a helpful AI assistant named SmolLM, trained by Hugging
Face". This template fills the gap with its own text.

In [7]:
no_system = tokenizer.apply_chat_template(
    messages[1:], tokenize=False,
    add_generation_prompt=True)
print(repr(no_system))

'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWhen do I use ser and when estar?<|im_end|>\n<|im_start|>assistant\n'


## 8. The tokens the model really gets

With `tokenize=True` you get a dict whose `input_ids` holds the token ids. That is what you feed the model.

**Look for:** the keys `input_ids` and `attention_mask`, **36** tokens, and `<|im_start|>` as one single token at
the start (it is a special token, not 12 characters). `Ċ` is how this tokenizer shows a newline and `Ġ` a space.

In [8]:
enc = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True)
print("returns:", type(enc).__name__, list(enc.keys()))
ids = enc["input_ids"]
print("token count:", len(ids))
print("first tokens:", tokenizer.convert_ids_to_tokens(ids)[:6])

returns: BatchEncoding ['input_ids', 'attention_mask']
token count: 36
first tokens: ['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'Ġa']


## 9. The library string is the recorded prompt

Build the gracias request from step 4 with `apply_chat_template` and compare it with `G_T`, the exact string the
lesson sent to the three models.

**Look for:** `same string as G_T: True`. So the replies A, C and D in step 4 came from exactly what the library
builds. Hand-written markers are only safe when they match the template byte for byte.

In [9]:
G_SYSTEM = "You are a patient language tutor. Answer in one or two sentences."
g_prompt = tokenizer.apply_chat_template(
    [{"role": "system", "content": G_SYSTEM},
     {"role": "user", "content": G_Q}],
    tokenize=False, add_generation_prompt=True)
print("same string as G_T:", g_prompt == G_T)

same string as G_T: True


## 10. The base model has no template at all

The base model's tokenizer is `HuggingFaceTB/SmolLM2-135M`. This cell asks whether it has a chat template, then
tries to build a chat prompt with it anyway. The call is **meant to fail**; the cell catches the error and prints it.

**Look for:** `has a chat template: False`, then a `ValueError` that says `tokenizer.chat_template is not set`.
Serving tools that read the template from the tokenizer fail the same way. A check like this would have caught
the app's swap before any learner saw it.

In [10]:
base_tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
print("has a chat template:", base_tok.chat_template is not None)
try:
    base_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
except Exception as err:
    print(type(err).__name__ + ":", str(err).split("!")[0] + "!")

has a chat template: False
ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed!


## 11. Broken on purpose: the right model, called the wrong way

The team switched to the tuned model, but their server call used `raw: true` with the bare question, so no turn
markers reached the model. Then they fixed the prompt with `apply_chat_template`.

**Look for:** sent bare, the tuned model writes a multiple-choice sheet ("Options are: ...") after 24 tokens, and
for two other bare questions it ends at once with an empty reply (1 token). With the template it replies (44 tokens). The
reply is still wrong: "Yo soy cansado" should use estar. The template fixes the shape of the answer, not what a
model this small knows.

In [11]:
YO_Q = "Is this sentence correct: Yo soy cansado?"
show("instruct, bare question (the bug)", generate(MODELS["instruct"], YO_Q))
for q in [question, G_Q]:
    show(f"instruct, bare question: {q}", generate(MODELS["instruct"], q))

fixed = tokenizer.apply_chat_template(
    [{"role": "system", "content": G_SYSTEM}, {"role": "user", "content": YO_Q}],
    tokenize=False, add_generation_prompt=True)
show("instruct, chat template (the fix)", generate(MODELS["instruct"], fixed))

--- instruct, bare question (the bug)
Options are:
1. No, it's not correct.
2. Yes, it is correct.
[done_reason: stop, tokens: 24]

--- instruct, bare question: When do I use ser and when estar?
(empty reply)
[done_reason: stop, tokens: 1]

--- instruct, bare question: What does gracias mean?
(empty reply)
[done_reason: stop, tokens: 1]

--- instruct, chat template (the fix)
Yes, that's the correct answer. The phrase "yoy soy" is an idiomatic expression used to express the idea of something being good or pleasant. It means "I have a good taste in mind."
[done_reason: stop, tokens: 44]



## Now change it

Each cell below has one line marked `# <- change this`. Edit it, run the cell, and compare.

### Experiment 1: drop the template

The cell sends the gracias question to the tuned model, with or without its chat template.

**What should change:** with `True` you get reply A, which stops on its own after 51 tokens. With `False` the
tuned model reads a bare question with no open assistant turn and ends at once: an empty reply, 1 token. The model
is the same; only the string changed.

In [12]:
use_template = True  # <- change this (try False)
p = g_prompt if use_template else G_Q
show(f"instruct, use_template={use_template}", generate(MODELS["instruct"], p))

--- instruct, use_template=True
Gracias means "thank you" in Spanish, and it's used to express gratitude towards someone for their kindness, support, or assistance. It can also be translated as "good luck," "fortunate circumstance," or simply "thankful."
[done_reason: stop, tokens: 51]



### Experiment 2: your own system message, or none

The cell builds the tutor request with the system message you choose.

**What should change:** your text appears in the first turn. With `None` the cell sends only the user message,
and the template's own line about "SmolLM, trained by Hugging Face" comes back. The template can add text you
never wrote.

In [13]:
system = "You are a friendly tutor named Lola."  # <- change this (try None, or your own text)
msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": ask}]
print(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

<|im_start|>system
You are a friendly tutor named Lola.<|im_end|>
<|im_start|>user
When do I use ser and when estar?<|im_end|>
<|im_start|>assistant



### Experiment 3: a marker is one token, a lookalike is not

The cell tokenizes a piece of text on its own.

**What should change:** `<|im_end|>` is one token with id 2, the end-of-turn token the model learned to write.
Change it to `im_end` or `<im_end>` and it falls apart into 3 or 5 ordinary pieces. A model from another family,
with other markers, sees your hand-written markers like this: as plain text.

In [14]:
text = "<|im_end|>"  # <- change this (try "im_end" or "<im_end>")
ids = tokenizer(text, add_special_tokens=False)["input_ids"]
print("tokens:", tokenizer.convert_ids_to_tokens(ids))
print("ids:", ids)

tokens: ['<|im_end|>']
ids: [2]


## What you should see

- Step 5: the printed prompt starts with `<|im_start|>system` and ends with an open `<|im_start|>assistant` line,
  with nothing after it.
- Step 6: with `add_generation_prompt=False` the last line is the user's turn:
  `<|im_start|>user`, then `When do I use ser and when estar?<|im_end|>`.
- Step 7: with only the user message, a system line you did not write appears: "You are a helpful AI assistant
  named SmolLM, trained by Hugging Face".
- Step 8: the tokenized prompt has 36 tokens, and the first one is `<|im_start|>` as a single token.
- Step 10: the base model's tokenizer prints `False` when asked whether it has a chat template.
- Steps 3 and 4: the base model's replies end with `done_reason: length` after 120 tokens; the tuned models'
  replies in the chat format end with `stop`.